# Setup

In [1]:
# Must run on new server launch
#!pip install -r requirements.txt

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not ((repo_root / "Final").exists() and (repo_root / "Sprint 3").exists()):
    if repo_root.parent == repo_root:
        raise RuntimeError("Could not locate repo root.")
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dataclasses import asdict, dataclass, field
from pathlib import Path
import json
from datetime import datetime
from typing import Any
import pandas as pd
from pprint import pprint, pp
from tqdm.auto import tqdm

from Final.config import default_config
from Final.paths import FINAL_ROOT
from Final.shared_utils import setup_logging, get_logger

from Final.models import (
    ExperimentState,
)
from Final.experiment_controller import ExperimentController
from Final.grid_search import GridSearchController
from Final.work_unit_scheduler import WorkUnitScheduler
from Final.coordination import CoordinationManager
from Final.pipeline_runtime import execute_pipeline_section
from Final.artifact_store import (
    LocalArtifactStore,
    DriveRegistryArtifactStore,
    HybridArtifactStore,
)
from Final.gating import (
    evaluate_module_card,
    decide_module_status,
    module_cards_to_frame,
)

from Final.labeling.pipeline import LabelingPipeline, LabelingPipelineConfig

In [2]:
cfg = default_config()

logger = setup_logging(
    name="shrub",
    log_dir=cfg.output.logs_root,
    log_filename="main_pipeline.log",
    force=True,
)

logger.info("Initialized main pipeline notebook.")
logger.info("FINAL_ROOT = %s", FINAL_ROOT)

MAIN_OUTPUT_ROOT = cfg.output.root / "main"
MAIN_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MAIN_ROOT = MAIN_OUTPUT_ROOT

MAIN_MANIFEST_DIR = MAIN_OUTPUT_ROOT / "manifests"
MAIN_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS_ROOT = MAIN_ROOT / "experiments"
EXPERIMENTS_ROOT.mkdir(parents=True, exist_ok=True)

2026-04-16 22:43:10 | INFO     | shrub | Initialized main pipeline notebook.
2026-04-16 22:43:10 | INFO     | shrub | FINAL_ROOT = /home/jovyan/work/Dry-shRub/shrub/Final


In [3]:
controller = ExperimentController(
    experiment_root=cfg.output.root / "experiments",
    experiment_name="shrubwise_main",
)

state = controller.load_state()

pipelines = {}

local_store = LocalArtifactStore(
    repo_root=cfg.data.project_root,
    storage_root=cfg.output.root / "artifact_store_local",
)

USE_DRIVE = True

if USE_DRIVE:
    drive_store = DriveRegistryArtifactStore(
        repo_root=cfg.data.project_root,
        registry_path=cfg.output.root / "artifact_registry.yaml",
        drive_config_path=cfg.data.project_root / "drive_config.yaml",
        client_secrets_path=cfg.data.project_root / "client_secrets.json",
        credentials_path=cfg.data.project_root / "pydrive_credentials.json",
    )
    artifact_store = HybridArtifactStore(
        local_store=local_store,
        remote_store=drive_store,
    )
else:
    artifact_store = local_store

grid = GridSearchController(controller=controller)

# Labeling

In [16]:
labeling_cfg = LabelingPipelineConfig(
    sprint3_variant="revised",
    sprint3_variants=("original", "revised"),
    run_sprint3=True,
    max_ptx_per_site=1,
    force_rerun_sprint3=False,
    require_success_artifacts_sprint3=True,
    cleanup_ptx_after_all_variants=True,
    cleanup_stale_ptx_before_run=True,
    stale_ptx_days=2,
    use_shape_descriptors=True,
    use_temporal_confidence=True,
    use_boundary_confidence=True,
    boundary_confidence_mode="universal",
    use_transform_confidence=False,
    use_object_subspace_filter=False,
    rasterization_mode="circle",
    multires=cfg.raster.create_multires,
    site_reference_dates={},
    subspace_min_component_pixels=4,
    subspace_min_object_confidence=0.55,
    subspace_min_transform_confidence=0.50,
    subspace_min_temporal_confidence=0.40,
    subspace_max_height_m=3.5,
    force_rerun_sprint4=False,
    force_refresh_site_assets=True,
    nonfatal_qa_overlay=True,
    allow_adopt_global_outputs=False,
)

cfg.labeling_runtime.storage.enable_local_store = True
cfg.labeling_runtime.storage.enable_drive_store = True
cfg.labeling_runtime.storage.use_hybrid_store = True

cfg.labeling_runtime.storage_policy.push_large_artifacts_to_remote = True
cfg.labeling_runtime.storage_policy.prune_local_after_remote_push = True
cfg.labeling_runtime.storage_policy.verify_remote_before_prune = True

labeling_pipeline = LabelingPipeline(cfg, pipeline_config=labeling_cfg)
base_pipeline = LabelingPipeline(cfg, pipeline_config=labeling_cfg)

pipelines["labeling"] = labeling_pipeline

print("Artifact store type:", type(labeling_pipeline.artifact_store).__name__)
print("Local store enabled:", cfg.labeling_runtime.storage.enable_local_store)
print("Drive store enabled:", cfg.labeling_runtime.storage.enable_drive_store)
print("Hybrid mode:", cfg.labeling_runtime.storage.use_hybrid_store)

2026-04-16 23:56:14 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-16 23:56:14 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
Artifact store type: HybridArtifactStore
Local store enabled: True
Drive store enabled: True
Hybrid mode: True


In [17]:
storage_cfg = cfg.artifact_store

print("client_secrets exists:", Path(storage_cfg.drive_client_secrets_path).exists(), storage_cfg.drive_client_secrets_path)
print("drive_config exists:", Path(storage_cfg.drive_config_path).exists(), storage_cfg.drive_config_path)
print("credentials exists:", Path(storage_cfg.drive_client_secrets_path).exists(), storage_cfg.drive_client_secrets_path)
print("drive_registry exists:", Path(storage_cfg.drive_registry_path).exists(), storage_cfg.drive_registry_path)

store = labeling_pipeline.artifact_store
print("Resolved artifact store:", type(store).__name__)

if hasattr(store, "local_store"):
    print("Hybrid local store root:", store.local_store.storage_root)
if hasattr(store, "remote_store"):
    print("Hybrid remote store type:", type(store.remote_store).__name__)
elif type(store).__name__ != "LocalArtifactStore":
    print("Remote store active:", type(store).__name__)

client_secrets exists: True /home/jovyan/work/Dry-shRub/shrub/client_secrets.json
drive_config exists: True /home/jovyan/work/Dry-shRub/shrub/drive_config.yaml
credentials exists: True /home/jovyan/work/Dry-shRub/shrub/client_secrets.json
drive_registry exists: True /home/jovyan/work/Dry-shRub/shrub/Final/artifact_registry.yaml
Resolved artifact store: HybridArtifactStore
Hybrid local store root: /home/jovyan/work/Dry-shRub/shrub/Final/artifact_store_local
Hybrid remote store type: DriveRegistryArtifactStore


In [18]:
rr = labeling_pipeline.runtime_report()
pprint(rr)

for stage_name in ["sprint3", "standardize", "refine", "transfer", "rasterize"]:
    ok, elig = labeling_pipeline.stage_is_eligible(stage_name, runtime_report=rr)
    pprint((stage_name, ok, {k: v.status.value for k, v in elig.items()}))

#display(labeling_pipeline.pipeline_spec)
display(grid.pipeline_module_state_frame(labeling_pipeline))

RuntimeCapabilityReport(detected_image_key='shrubs-labels-v1',
                        detected_image_alias='pramonettivega/shrubs-labels:v1',
                        detected_conda_env='base',
                        capabilities=['runtime:features',
                                      'runtime:labeling_transfer',
                                      'runtime:modeling',
                                      'runtime:pdal',
                                      'runtime:python',
                                      'runtime:rasterio'],
                        available_executables=['python', 'pdal'],
                        available_python_modules=['numpy',
                                                  'pandas',
                                                  'rasterio',
                                                  'scipy'],
                        marker_files_found=[],
                        marker_env_matches={'JUPYTER_IMAGE_SPEC': 'pramonettivega/shrubs-labels:v1'}

,pipeline,stage_name,module_name,enabled,variant_name,params
0,labeling,sprint3,labeling.sprint3.execution,True,"('original', 'revised')","{'max_ptx_per_site': 1, 'force_rerun_sprint3':..."
1,labeling,standardize,labeling.standardize.base,True,default,{}
2,labeling,refine,labeling.refine.shape_descriptors,True,enabled,{}
3,labeling,refine,labeling.refine.temporal_confidence,True,enabled,{'site_reference_dates': {}}
4,labeling,refine,labeling.refine.object_subspace_filter,False,disabled,"{'subspace_min_object_confidence': 0.55, 'subs..."
5,labeling,transfer,labeling.transfer.base,True,default,{}
6,labeling,rasterize,labeling.rasterize.mode,True,circle,{}
7,labeling,rasterize,labeling.boundary_confidence,True,universal,{}
8,labeling,rasterize,labeling.mask_subspace_reduction,False,disabled,{'subspace_min_component_pixels': 4}
9,labeling,rasterize,labeling.multires_export,True,default,"{'multires': (1.0, 2.0, 5.0, 10.0)}"


In [19]:
labeling_space_df = grid.section_space_frame(labeling_pipeline).copy()
display(labeling_space_df)
print("Total labeling variants:", len(labeling_space_df))

,config_signature,sprint3_variants,use_temporal_confidence,boundary_confidence_mode,use_object_subspace_filter,max_ptx_per_site
0,01236fe9e16c,"original,revised",False,radial,False,1
1,0bab4176c29e,"original,revised",True,universal,False,1
2,15a87fd1fa6a,revised,True,radial,False,1
3,2a5857f983d5,revised,False,radial,True,1
4,3628fb5a1777,"original,revised",True,radial,False,1
5,5960d3bed0d8,revised,False,universal,False,1
6,69512a22f6c7,"original,revised",True,radial,True,1
7,7568adeff82a,revised,True,universal,False,1
8,801b3fe64605,"original,revised",False,universal,False,1
9,8475fdd27e30,revised,True,radial,True,1


Total labeling variants: 16


## One Run

In [22]:
TRIAL_ID = "labeling_only_trial_001"

trial_path = controller.trial_path(TRIAL_ID)
if trial_path.exists():
    trial = controller.load_trial(TRIAL_ID)
else:
    trial = controller.create_trial(trial_id=TRIAL_ID)

controller.set_section_config(
    trial,
    "labeling",
    labeling_pipeline.config_dict(),
)
controller.save_trial(trial)
#trial

PosixPath('/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/experiments/shrubwise_main/trials/labeling_only_trial_001.json')

In [23]:
labeling_result, state, runtime_stats = execute_pipeline_section(
    labeling_pipeline,
    state=state,
    artifact_store=artifact_store,
    push_remote=USE_DRIVE,
)

controller.record_section_result(
    trial,
    section_name="labeling",
    config_signature=labeling_pipeline.config_signature(),
    result=labeling_result,
)
controller.save_trial(trial)
controller.save_state(state)

labeling_result, runtime_stats

[artifact_store] Using cached Google Drive credentials.
2026-04-16 21:34:10 | INFO     | shrub.labeling.pipeline | JSON ARTIFACT MISS | key=run_manifest | rel_path=labeling/0bab4176c29e/run_manifest.json
2026-04-16 21:34:10 | INFO     | shrub.labeling.pipeline | Skipping stage=sprint3 | reason=labeling.sprint3.execution: missing=['runtime:intelimon_sprint3']
2026-04-16 21:34:11 | INFO     | shrub.labeling.pipeline | Using module-aware standardize cache | data=3228b0743a1a2741 | config=a368033307bb3009
2026-04-16 21:34:11 | INFO     | shrub.labeling.pipeline | STORAGE POLICY RECONCILE | stage=standardize | cache_dir=/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/labeling/stage_cache/standardize/3228b0743a1a2741__a368033307bb3009 | n_artifacts=1
2026-04-16 21:34:11 | INFO     | shrub.labeling.pipeline | STORAGE POLICY RECONCILE DONE | stage=standardize | n_artifacts=1 | statuses={'objects_csv': 'reconciled_local_path'}
2026-04-16 21:34:11 | INFO     | shrub.labeling.pipeline | Using m

(PipelineRunResult(pipeline_name='labeling', success=True, status='success', raster_outputs=CanonicalRasterOutputs(labels=                  site_id                plot_id    plot_key   variant  \
 0     calaveras-big-trees  CATCU_0009_20250615_1  CATCU_0009  original   
 4     calaveras-big-trees  CATCU_0009_20250615_1  CATCU_0009   revised   
 1     calaveras-big-trees  CATCU_0009_20250615_1  CATCU_0009  original   
 5     calaveras-big-trees  CATCU_0009_20250615_1  CATCU_0009   revised   
 2     calaveras-big-trees  CATCU_0009_20250615_1  CATCU_0009  original   
 6     calaveras-big-trees  CATCU_0009_20250615_1  CATCU_0009   revised   
 3     calaveras-big-trees  CATCU_0009_20250615_1  CATCU_0009  original   
 7     calaveras-big-trees  CATCU_0009_20250615_1  CATCU_0009   revised   
 8                dl-bliss  CAAEU_0027_20250728_1  CAAEU_0027  original   
 12               dl-bliss  CAAEU_0027_20250728_1  CAAEU_0027   revised   
 9                dl-bliss  CAAEU_0027_20250728_1  CAA

In [ ]:
# stderr_path = "/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/labeling/calaveras-big-trees/sprint3/original/CATCU_0009_20250615_1/stderr.log"
# stdout_path = "/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/labeling/calaveras-big-trees/sprint3/original/CATCU_0009_20250615_1/stdout.log"

# print("================ FULL STDERR ================")
# if os.path.exists(stderr_path):
#     with open(stderr_path, 'r') as f:
#         print(f.read())
# else:
#     print("stderr.log not found!")

# print("\n================ FULL STDOUT ================")
# if os.path.exists(stdout_path):
#     with open(stdout_path, 'r') as f:
#         print(f.read())
# else:
#     print("stdout.log not found!")

In [24]:
#print("Experiment state:")
#display(state)

# print("Trial:")
# display(trial)

print("Completed trials:")
display(grid.completed_trials_frame())

Completed trials:


,trial_id,status,n_section_runs
0,labeling_only_trial_001,in_progress,1


## Full Run

In [20]:
coordination = CoordinationManager(
    artifact_store=base_pipeline.artifact_store,
    root_prefix=cfg.coordination.root_prefix,
)
scheduler = WorkUnitScheduler(controller=controller, coordination=coordination)

config_space = base_pipeline.enumerate_config_space()

len(config_space), config_space[0]

(16,
 {'sprint3_variant': 'revised',
  'use_shape_descriptors': True,
  'use_temporal_confidence': False,
  'use_boundary_confidence': True,
  'use_transform_confidence': False,
  'use_object_subspace_filter': False,
  'rasterization_mode': 'circle',
  'multires': (1.0, 2.0, 5.0, 10.0),
  'force_rerun_sprint4': False,
  'force_refresh_site_assets': True,
  'nonfatal_qa_overlay': True,
  'run_sprint3': True,
  'sprint3_variants': ('revised',),
  'max_ptx_per_site': 1,
  'force_rerun_sprint3': False,
  'require_success_artifacts_sprint3': True,
  'cleanup_ptx_after_all_variants': True,
  'cleanup_stale_ptx_before_run': True,
  'stale_ptx_days': 2,
  'boundary_confidence_mode': 'radial',
  'site_reference_dates': {},
  'subspace_min_component_pixels': 4,
  'subspace_min_object_confidence': 0.55,
  'subspace_min_transform_confidence': 0.5,
  'subspace_min_temporal_confidence': 0.4,
  'subspace_max_height_m': 3.5,
  'allow_adopt_global_outputs': False})

In [21]:
trials = []
pipelines_by_trial = {}

for config_dict in config_space:
    pipeline_cfg = LabelingPipelineConfig(**config_dict)
    pipeline = LabelingPipeline(cfg, pipeline_config=pipeline_cfg)

    trial = grid.get_or_create_trial_for_config(
        pipeline=pipeline,
        config_dict=config_dict,
        trial_prefix="labeling_trial",
    )

    trials.append(trial)
    pipelines_by_trial[trial.trial_id] = {"labeling": pipeline}

pipelines_by_trial

2026-04-16 23:56:34 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-16 23:56:34 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-16 23:56:35 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-16 23:56:35 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-16 23:56:35 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-16 23:56:36 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-16 23:56:36 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-16 23:56:36 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-16 23:56:36 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-16 23:56:37 | INFO     | shru

{'labeling_trial_9b7b2b083481': {'labeling': <Final.labeling.pipeline.LabelingPipeline at 0x7fa25d334740>},
 'labeling_trial_2a5857f983d5': {'labeling': <Final.labeling.pipeline.LabelingPipeline at 0x7fa25d334e30>},
 'labeling_trial_5960d3bed0d8': {'labeling': <Final.labeling.pipeline.LabelingPipeline at 0x7fa25d3358b0>},
 'labeling_trial_b8a727c60919': {'labeling': <Final.labeling.pipeline.LabelingPipeline at 0x7fa25d2cd9a0>},
 'labeling_trial_15a87fd1fa6a': {'labeling': <Final.labeling.pipeline.LabelingPipeline at 0x7fa25d334e60>},
 'labeling_trial_8475fdd27e30': {'labeling': <Final.labeling.pipeline.LabelingPipeline at 0x7fa25d3347d0>},
 'labeling_trial_7568adeff82a': {'labeling': <Final.labeling.pipeline.LabelingPipeline at 0x7fa25d4127e0>},
 'labeling_trial_be110217cd7d': {'labeling': <Final.labeling.pipeline.LabelingPipeline at 0x7fa25d337770>},
 'labeling_trial_01236fe9e16c': {'labeling': <Final.labeling.pipeline.LabelingPipeline at 0x7fa25d344200>},
 'labeling_trial_a8866b4d1f6

In [22]:
progress_bar = tqdm(total=len(trials), desc="Resolved trials")

def progress_callback(trials, pipelines, active_job=None):
    resolved = 0
    for t in trials:
        latest = controller.load_trial(t.trial_id)
        if latest.status in {"success", "failed"}:
            resolved += 1
    progress_bar.n = resolved
    progress_bar.refresh()

while True:
    # flatten trial->pipeline mapping into the shape scheduler expects
    active_trials = [controller.load_trial(t.trial_id) for t in trials]

    # trial-local pipeline selection for now: one pipeline per trial
    next_job = None
    candidates = []
    for trial in active_trials:
        pl_map = pipelines_by_trial[trial.trial_id]
        selected = scheduler.select_next_job(trials=[trial], pipelines=pl_map)
        if selected is not None:
            candidates.append(selected)

    if not candidates:
        progress_callback(active_trials, {}, active_job=None)
        break

    # global selection
    candidates = sorted(
        candidates,
        key=lambda x: (
            x[2].get("priority", 100),
            x[0].trial_id,
            x[2].get("unit_id"),
        ),
    )
    trial, pipeline, unit = candidates[0]

    claimed, _ = scheduler.claim_unit(trial=trial, pipeline=pipeline, unit=unit)
    if not claimed:
        continue

    scheduler.run_claimed_job(
        trial=trial,
        pipeline=pipeline,
        unit=unit,
        state=state,
    )

    controller.save_state(state)
    progress_callback(active_trials, {}, active_job=(trial, pipeline, unit))

progress_bar.close()

Resolved trials:   0%|          | 0/16 [00:00<?, ?it/s]

[artifact_store] Refreshing expired Google Drive credentials.
[artifact_store] Creating new Drive artifact for coordination/shrubwise_main/labeling_trial_01236fe9e16c/labeling/transfer/calaveras-big-trees|CATCU_0009_20250615_1|sprint3_original.json
[artifact_store] Updating existing Drive artifact for coordination/shrubwise_main/labeling_trial_01236fe9e16c/labeling/transfer/calaveras-big-trees|CATCU_0009_20250615_1|sprint3_original.json
2026-04-16 23:57:15 | INFO     | shrub.labeling.pipeline | Using module-aware standardize cache | data=3228b0743a1a2741 | config=a368033307bb3009
2026-04-16 23:57:15 | INFO     | shrub.labeling.pipeline | STORAGE POLICY RECONCILE | stage=standardize | cache_dir=/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/labeling/stage_cache/standardize/3228b0743a1a2741__a368033307bb3009 | n_artifacts=1
2026-04-16 23:57:15 | INFO     | shrub.labeling.pipeline | STORAGE POLICY RECONCILE DONE | stage=standardize | n_artifacts=1 | statuses={'objects_csv': 'reconciled

KeyboardInterrupt: 

In [ ]:
display(grid.trials_frame())
display(grid.completed_trials_frame())

# Features

In [ ]:
features_cfg = {
    "enabled": False,
    "feature_families": [],
    "notes": "Placeholder only for now.",
}

trial.features_config = features_cfg

save_trial(EXPERIMENT_NAME, trial)

trial.features_config

In [ ]:
trial.features_result = {
    "success": None,
    "status": "placeholder_not_run",
    "metrics": {},
    "qa_outputs": {"placeholder": True},
    "notes": ["Features pipeline not implemented yet."],
}

state.section_status["features"] = "placeholder_not_run"
state.qa_outputs["features"] = trial.features_result["qa_outputs"]

save_trial(EXPERIMENT_NAME, trial)
trial.features_result

# Modeling

In [ ]:
modeling_cfg = {
    "enabled": False,
    "model_family": None,
    "notes": "Placeholder only for now.",
}

trial.modeling_config = modeling_cfg

save_trial(EXPERIMENT_NAME, trial)

trial.modeling_config

In [ ]:
trial.modeling_result = {
    "success": None,
    "status": "placeholder_not_run",
    "metrics": {},
    "qa_outputs": {"placeholder": True},
    "notes": ["Modeling pipeline not implemented yet."],
}

state.section_status["modeling"] = "placeholder_not_run"
state.qa_outputs["modeling"] = trial.modeling_result["qa_outputs"]

save_trial(EXPERIMENT_NAME, trial)
trial.modeling_result

# Post-processing

In [ ]:
postprocessing_cfg = {
    "enabled": False,
    "steps": [],
    "notes": "Placeholder only for now.",
}

trial.postprocessing_config = postprocessing_cfg

save_trial(EXPERIMENT_NAME, trial)

trial.postprocessing_config

In [ ]:
trial.postprocessing_result = {
    "success": None,
    "status": "placeholder_not_run",
    "metrics": {},
    "qa_outputs": {"placeholder": True},
    "notes": ["Postprocessing pipeline not implemented yet."],
}

state.section_status["postprocessing"] = "placeholder_not_run"
state.qa_outputs["postprocessing"] = trial.postprocessing_result["qa_outputs"]

save_trial(EXPERIMENT_NAME, trial)
trial.postprocessing_result

# Finalize Trial

In [ ]:
trial.qa_summary = {
    "integrity": {
        "labeling": trial.labeling_result.get("status"),
        "features": trial.features_result.get("status"),
        "modeling": trial.modeling_result.get("status"),
        "postprocessing": trial.postprocessing_result.get("status"),
    },
    "section_level": {
        "labeling": "placeholder",
        "features": "placeholder",
        "modeling": "placeholder",
        "postprocessing": "placeholder",
    },
    "cross_section": {
        "label_to_model_feedback": "placeholder",
        "feature_to_model_feedback": "placeholder",
        "end_to_end_feedback": "placeholder",
    },
}

save_trial(EXPERIMENT_NAME, trial)
trial.qa_summary

In [ ]:
labeling_object_rows = trial.labeling_result.get("metrics", {}).get("n_object_rows", 0)
labeling_artifact_rows = trial.labeling_result.get("metrics", {}).get("n_artifact_rows", 0)

trial.score_summary = {
    "labeling_score_placeholder": float(labeling_object_rows > 0) + float(labeling_artifact_rows > 0),
    "features_score_placeholder": None,
    "modeling_score_placeholder": None,
    "postprocessing_score_placeholder": None,
    "composite_score_placeholder": float(labeling_object_rows > 0) + float(labeling_artifact_rows > 0),
}

save_trial(EXPERIMENT_NAME, trial)
trial.score_summary

In [ ]:
trial.status = "partial" if (
    trial.features_result.get("status") == "placeholder_not_run"
    or trial.modeling_result.get("status") == "placeholder_not_run"
    or trial.postprocessing_result.get("status") == "placeholder_not_run"
) else "completed"

save_trial(EXPERIMENT_NAME, trial)

trial

In [ ]:
trial_record_min = {
    "trial_id": trial.trial_id,
    "created_at": trial.created_at,
    "status": trial.status,
    "labeling_config": trial.labeling_config,
    "features_config": trial.features_config,
    "modeling_config": trial.modeling_config,
    "postprocessing_config": trial.postprocessing_config,
    "score_summary": trial.score_summary,
}

registry.trials = [t for t in registry.trials if t["trial_id"] != trial.trial_id]
registry.trials.append(trial_record_min)

score = trial.score_summary.get("composite_score_placeholder")
if score is not None:
    if registry.best_score is None or score > registry.best_score:
        registry.best_score = score
        registry.best_trial_id = trial.trial_id

save_registry(registry)

registry

In [ ]:
trials_df = pd.DataFrame(registry.trials)

if not trials_df.empty:
    if "score_summary" in trials_df.columns:
        trials_df["composite_score_placeholder"] = trials_df["score_summary"].apply(
            lambda x: x.get("composite_score_placeholder") if isinstance(x, dict) else None
        )

    display(
        trials_df[
            ["trial_id", "created_at", "status", "composite_score_placeholder"]
        ].sort_values("trial_id")
    )

    print("Best trial:", registry.best_trial_id)
    print("Best score:", registry.best_score)
else:
    print("No trials recorded yet.")

In [ ]:
INSPECT_TRIAL_ID = trial.trial_id  # change manually

inspect_trial = load_trial(EXPERIMENT_NAME, INSPECT_TRIAL_ID)
inspect_trial